# Negative control tests

In this notebook, we generate "negative control" evaluations of the validation method by taking the examined burden scores and computing ORs against mismatched phenotype sets.

Our gene-associated diseases can be split broadly into five categories:
- Cancer
- Long QT
- MODY,
- Hearing Loss and
- Hypercholesterolemia

We use these categories to create collections of mismatched ORs by matching each gene in one category to cohorts characteristic of the other four categories.

In [1]:
genes_by_cat = {
    'Cancer': {
        'BAP1',
        'BARD1',
        'BRCA1',
        'BRCA2',
        'CHEK2',
        'MSH2',
        'PALB2',
        'PTEN',
        'RAD51C',
        'RAD51D',
        'TP53',
        'TSC2'
    },
    'MODY': {'GCK'},
    'Long QT': {
        'KCNE1',
        'KCNH2',
        'SCN5A',
    },
    'Hearing Loss': {'KCNQ4'},
    'Hypercholesterolemia': {'LDLR'}
}

While our cohorts for individual cancer genes vary, we select the broadest cancer case cohort for this category.

In [2]:
# Representative cohorts:
cohort_by_cat = {
    'Cancer': 'TP53',
    'MODY': 'GCK',
    'Long QT': 'KCNE1',
    'Hearing Loss': 'KCNQ4',
    'Hypercholesterolemia': 'LDLR'
}

# Gene to category mapping
gene_cats = {
    gene: cat
    for cat, genes in genes_by_cat.items()
    for gene in genes
}    

## OR estimates

In [3]:
import os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

In [4]:
# Constants (mostly paths to things)

functional_carriers_path = 'carrier_tables/functional-carriers_2026-10-07.parquet'
excalibr_carriers_path = 'carrier_tables/excalibr-carriers_2026-09-30.parquet'
vep_carriers_path = 'carrier_tables/vep-carriers_2026-09-29.parquet'
tp_carriers_path = 'carrier_tables/total-points-carriers_2026-10-01.parquet'

demographics_path = '../cohorts/demo.tsv.gz'
ancestry_pca_path = '../cohorts/ancestry_pca.tsv.gz'

functional_or_path = 'results/functional-NC-OR-estimates_2026-10-08.parquet'
excalibr_or_path = 'results/excalibr-NC-OR-estimates_2026-10-08.parquet'
vep_or_path = 'results/vep-NC-OR-estimates_2026-10-08.parquet'
tp_or_path = 'results/tp-NC-OR-estimates_2026-10-08.parquet'

# Index column labels
index_cols=['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

OR calculation

In [5]:
demographics_df = pd.read_table(demographics_path, index_col='person_id')
demographics_df

In [6]:
pca_df = pd.read_table(ancestry_pca_path, index_col='research_id')
pca_df

In [7]:
from functools import cache

@cache
def assemble_cohort(gene):
    result = pd.read_parquet(f'gene_cohorts/{gene} cohort.parquet').join(demographics_df).join(pca_df)
    # Trichotomize sex
    result.sex_at_birth = result.sex_at_birth.where(
        result.sex_at_birth.isin(['Male', 'Female']),
        'Other'
    )
    
    return result


In [8]:
from cvfgaou import odds

In [9]:
carrier_index_cols = ['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range', 's']
or_index_cols = [
    'Dataset/Predictor',
    'Gene',
    'Grouping/Calibration',
    'Consequence',
    'Group/Range',
    'Mismatched Phenotype'
]

for carriers_file, odds_file in (
    (functional_carriers_path, functional_or_path),
    #(excalibr_carriers_path, excalibr_or_path),
    (vep_carriers_path, vep_or_path),
    (tp_carriers_path, tp_or_path)
):

    carriers_df = pd.read_parquet(carriers_file)
    carriers_df.index.set_names(carrier_index_cols, inplace=True)

    or_estimates_dict = {
        (dataset, gene, grouping, consequence, group, mismatched_phenotype):
            odds.estimate_logOR(
                pd.to_numeric(exposure_df.index.to_frame().s, downcast='integer'),
                assemble_cohort(cohort_by_cat[mismatched_phenotype])
            )
        for (dataset, gene, grouping, consequence, group), exposure_df
        in tqdm(carriers_df.groupby(index_cols))
        for mismatched_phenotype, phenotype_genes in genes_by_cat.items()
        if gene not in phenotype_genes
    }

    or_estimates_df = pd.DataFrame.from_dict(or_estimates_dict, orient='index')
    or_estimates_df.index.set_names(or_index_cols, inplace=True)
    or_estimates_df.to_parquet(odds_file)
